In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator , TransformerMixin
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.model_selection import cross_validate
from sklearn.compose import ColumnTransformer

from sklearn.model_selection import RandomizedSearchCV
from imblearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from dataclasses import dataclass
from sklearn.model_selection import KFold
from imblearn.pipeline import pipeline
import scipy
from MlUtlitys import PlotManager

ModuleNotFoundError: No module named 'MlUtlitys'

In [ ]:
linux_path = r"/run/media/drdrakken/Elements/Sonstiges/Programmieren/Machine Learning/csvs/titanic.csv"
windows_path = r"I:\Sonstiges\Programmieren\Machine Learning/csvs/titanic.csv"
synthea_location = r"/home/drdrakken/projekte/synthea/output/fhir/ausgabe/Location.csv"
synthea_organization = r"/home/drdrakken/projekte/synthea/output/fhir/ausgabe/Organization.csv"
df = pd.read_csv(synthea_organization)

In [ ]:
df

,resourceType,id,extension,identifier,active,type,name,telecom,address,meta.profile
0,Organization,74ab949d-17ac-3309-83a0-13b4405c66aa,[{'url': 'http://synthetichealth.github.io/syn...,[{'system': 'https://github.com/synthetichealt...,True,[{'coding': [{'system': 'http://terminology.hl...,Fitchburg Outpatient Clinic,"[{'system': 'phone', 'value': '978-342-9781 Or...","[{'line': ['881 Main Street'], 'city': 'Fitchb...",['http://hl7.org/fhir/us/core/StructureDefinit...
1,Organization,5ba6765a-f335-36b0-9c6a-6bb494c5330d,[{'url': 'http://synthetichealth.github.io/syn...,[{'system': 'https://github.com/synthetichealt...,True,[{'coding': [{'system': 'http://terminology.hl...,"URGENT CARE MEDICAL ASSOCIATES, LLC","[{'system': 'phone', 'value': '2038850808'}]","[{'line': ['31 OLD ROUTE 7'], 'city': 'WESTWOO...",['http://hl7.org/fhir/us/core/StructureDefinit...
2,Organization,3b021cb6-4745-3d9d-8436-230055f770bd,[{'url': 'http://synthetichealth.github.io/syn...,[{'system': 'https://github.com/synthetichealt...,True,[{'coding': [{'system': 'http://terminology.hl...,"BETH ISRAEL DEACONESS HOSPITAL-NEEDHAM, INC.","[{'system': 'phone', 'value': '7814533000'}]","[{'line': ['148 CHESTNUT ST'], 'city': 'NEEDHA...",['http://hl7.org/fhir/us/core/StructureDefinit...
3,Organization,880c8804-deb0-35b0-9132-06950118b8b3,[{'url': 'http://synthetichealth.github.io/syn...,[{'system': 'https://github.com/synthetichealt...,True,[{'coding': [{'system': 'http://terminology.hl...,BRIARWOOD REHABILITATION & HEALTHCARE CENTER,"[{'system': 'phone', 'value': '7814494040'}]","[{'line': ['150 LINCOLN STREET'], 'city': 'NEE...",['http://hl7.org/fhir/us/core/StructureDefinit...
4,Organization,f52777bd-7313-3b27-8dd5-cd3f87bd3275,[{'url': 'http://synthetichealth.github.io/syn...,[{'system': 'https://github.com/synthetichealt...,True,[{'coding': [{'system': 'http://terminology.hl...,"WALTHAM WESTON INTERNAL MEDICINE ASSOCIATES, P.C.","[{'system': 'phone', 'value': '7815476030'}]","[{'line': ['9 HOPE AVE'], 'city': 'WESTWOOD', ...",['http://hl7.org/fhir/us/core/StructureDefinit...
5,Organization,acfde9fd-b6ef-36d0-86dd-ab77398d289b,[{'url': 'http://synthetichealth.github.io/syn...,[{'system': 'https://github.com/synthetichealt...,True,[{'coding': [{'system': 'http://terminology.hl...,HEBREW SENIORLIFE HOSPICE CARE INC,"[{'system': 'phone', 'value': '7812349805'}]","[{'line': ['80 NEWBRIDGE WAY'], 'city': 'DEDHA...",['http://hl7.org/fhir/us/core/StructureDefinit...


In [ ]:
@dataclass
class Config():
    target:str = "active"
    seed : int = 1234
    test_size:int = 0.2
    hyperparameter_test_amount:int = 5
    cross_validation_amount:int = 5 
    verbose:int = 0
    
config = Config()

In [ ]:
class DataClass():
    def __init__(self , DataFrame):
        self.data = DataFrame.copy()
        self.numerical_data = self.data.select_dtypes(include = np.number).columns
        self.categorical_data = self.data.select_dtypes(exclude = np.number).columns

        self.x = self.data.drop([config.target] , axis = 1)
        self.y = self.data[config.target]

data = DataClass(DataFrame = df)

In [ ]:
class DataTransform():

    def fit(self , X , y = None):
        x = X.copy()
        return x
    
    def transform(self, X , y = None):
        x = X.copy()
        x = self.new_features(x)
        return x
    
    def new_features(self , X , y = None):
        x = X.copy()
 
        return x

In [ ]:
class Preprocessor():

    def fit(self, X , y = None):

        self.preprocessor = ColumnTransformer([
            ("numerical_handler" , Pipeline([
                ("numerical_imputer" , SimpleImputer(strategy = "mean")),
                ("scaler" , MinMaxScaler()),
            ]),self.numerical_data),

            ("categorical_handler" , Pipeline([
                ("categorical_imputer" , SimpleImputer(strategy = "most_frequent")),
                ("categorical_encoder" , OneHotEncoder(handle_unknown="ignore" , sparse_output=False)),
            ]),self.categorical_data)
        ])
        self.preprocessor.fit(X)
        return self
    
    def transform(self , X , y = None):
        return self.preprocessor.transform(X)

In [ ]:
def model_varianz():

    return {

        "LogisticRegression":LogisticRegression(random_state = config.seed),
        "LinearSVC":LinearSVC(random_state = config.seed),
        "DecisionTreeClassifier":DecisionTreeClassifier(random_state = config.seed),
        "RandomForestClassifier":RandomForestClassifier(random_state = config.seed),
        "XGBClassifier":XGBClassifier(random_state = config.seed),
        
    }

In [ ]:
def custom_pipeline(estimator , transform):

    steps = []
    if transform:
        steps.append(("Transformed Performance" , DataTransform()))

    steps.append(("Base_Transformation_Score" , Preprocessor()))
    steps.append(("estimator" , estimator))
    return Pipeline(steps)

In [ ]:
def custom_scoring_params():
    return {

        "LogisticRegression": {
            "estimator__C": scipy.stats.loguniform(1e-4, 1e2),
            "estimator__penalty": ["l2"],
            "estimator__solver": ["lbfgs"],
        },

        "DecisionTreeClassifier": {
            "estimator__criterion": ["gini", "entropy"],
            "estimator__max_depth": [None, 5, 10, 20],
            "estimator__min_samples_split": scipy.stats.randint(2, 20),
            "estimator__min_samples_leaf": scipy.stats.randint(1, 10),
        },

        "RandomForestClassifier": {
            "estimator__n_estimators": scipy.stats.randint(100, 500),
            "estimator__max_depth": scipy.stats.randint(5, 40),
            "estimator__min_samples_split": scipy.stats.randint(2, 20),
            "estimator__min_samples_leaf": scipy.stats.randint(1, 10),
            "estimator__max_features": ["sqrt", "log2"],
            "estimator__bootstrap": [True, False],
        },

        "LinearSVC": {
            "estimator__C": scipy.stats.loguniform(1e-4, 1e2),
            "estimator__loss": ["hinge", "squared_hinge"],
            "estimator__dual": [True],
            "estimator__max_iter": [5000, 10000],
        },

        "XGBClassifier": {
            "estimator__n_estimators": scipy.stats.randint(100, 500),
            "estimator__learning_rate": scipy.stats.loguniform(1e-3, 0.3),
            "estimator__max_depth": scipy.stats.randint(3, 10),
            "estimator__subsample": scipy.stats.uniform(0.6, 0.4),
            "estimator__colsample_bytree": scipy.stats.uniform(0.6, 0.4),
            "estimator__gamma": scipy.stats.uniform(0, 5),
            "estimator__min_child_weight": scipy.stats.randint(1, 10),
        },

    }

In [ ]:
def custom_data_validation(estimator , X_train , y_train):
    
    k_fold = KFold(n_splits=5 , shuffle=True,random_state=config.seed)
    return cross_validate(estimator= estimator,
                          X=X_train,
                          y=y_train,
                          cv=k_fold,
                          n_jobs=config.hyperparameter_test_amount,
                          return_train_score=True,
                          scoring=custom_scoring_params(),
                          verbose=config.verbose,
                          )

In [ ]:
def custom_grid_search(estimator , scoring_dict):
    return RandomizedSearchCV(estimator= estimator,
                            cv=config.cross_validation_amount,
                            n_iter=config.hyperparameter_test_amount,
                            n_jobs=-1,
                            param_distributions=scoring_dict,
                            scoring=scoring_dict,
                            refit="accuracy",
                            return_train_score=True,
                            )

In [ ]:
def data_split():
    X_train , X_test , y_train , y_test = train_test_split(data.x,
                                                           data.y,
                                                           test_size=config.test_size,
                                                           shuffle=True,
                                                           stratify=data.y)
    print(f"X_train_shape:{X_train.shape}")
    print(f"X_test_shape:{X_test.shape}")
    print(f"y_train_shape:{y_train.shape}")
    print(f"y_test_shape:{y_test.shape}")

    return X_train , X_test , y_train , y_test

In [ ]:
def plot_scores(
        self,
        data,
        x_column,
        score_columns,
        title="Model Comparison",
        ylabel="Score"
    ):
        # Falls eine Liste von Dictionaries übergeben wird
        if isinstance(data, list):
            data = pd.DataFrame(data)

        x = range(len(data))

        plt.figure(figsize=(12, 6))

        for column in score_columns:
            if column in data.columns:
                plt.plot(
                    x,
                    data[column],
                    marker="o",
                    linewidth=2,
                    label=column
                )

        plt.xticks(x, data[x_column], rotation=45)
        plt.xlabel(x_column)
        plt.ylabel(ylabel)
        plt.title(title)
        plt.grid(True, linestyle="--", alpha=0.5)
        plt.legend()
        plt.tight_layout()
        plt.show()

In [ ]:
class Benchmark():

    def __init__(self):
        self.X_train , self.X_test , self.y_train , self.y_test = data_split()
        self.results = []

    def fit(self):

        for estimator_name , estimators in model_varianz().items():

            base_pipe = custom_pipeline(estimator=estimators , transform=False)
            transformer_pipe = custom_pipeline(estimator=estimators , transform=True)

            if self.use_cv:
                base_cv = custom_data_validation(estimator=base_pipe , 
                                                 X_train=self.X_train,
                                                 y_train=self.y_train)
                
                transformed_cv = custom_data_validation(estimator=transformer_pipe,
                                                        X_train=self.X_train,
                                                        y_train=self.y_train)

                self.results.append({

                    "Estimator" : estimators,

                    "Base_cv_train_score":base_cv["train_score"].mean(),
                    "Base_cv_test_score":base_cv["test_score"].mean(),
                    "Base_cv_std_score":base_cv["test_score"].std(),

                    "Transformed_cv_train_score":transformed_cv["train_score"].mean(),
                    "Transformed_cv_test_score":transformed_cv["test_score"].mean(),
                    "Transformed_cv_std_score":transformed_cv["test_score"].std(),
                    
                })

            if self.use_grid_search:
                base_grid = custom_grid_search(estimator=base_pipe,
                                               scoring_dict=custom_scoring_params()[estimator_name])
                
                transformed_grid = custom_grid_search(estimator=transformer_pipe,
                                               scoring_dict=custom_scoring_params()[estimator_name])

                self.results.append({

                    "base_best_estimator":base_grid.best_estimator_,
                    "transformed_best_estimator":transformed_grid.best_estimator_,
                    
                })
                
        plot_scores(
            data=self.results,
            x_column="Estimator",
            score_columns=[
                "Base_cv_train_score",
                "Base_cv_test_score",
                "Base_cv_std_score",
                "Transformed_cv_train_score",
                "Transformed_cv_test_score",
                "Transformed_cv_std_score",
            ],
            title="Cross Validation Results",
            ylabel="Score"
        )           
            
 

In [ ]:
Benchmark()

X_train_shape:(4, 9)
X_test_shape:(2, 9)
y_train_shape:(4,)
y_test_shape:(2,)
